<a href="https://colab.research.google.com/github/rafikreis/Previsao-de-acoes/blob/main/CESTA_STOCKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-optimize

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
import os
import joblib
import pickle
from datetime import datetime
import json
from tqdm import tqdm
import time
warnings.filterwarnings('ignore')

# ============================================================================
# FUNÇÃO PARA CONVERTER TIPOS NUMPY PARA PYTHON
# ============================================================================

def converter_para_python(obj):
    """Converte objetos numpy para tipos Python nativos para serialização JSON"""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: converter_para_python(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [converter_para_python(v) for v in obj]
    elif isinstance(obj, tuple):
        return tuple(converter_para_python(v) for v in obj)
    else:
        return obj

# ============================================================================
# CONFIGURAÇÕES GERAIS
# ============================================================================

# ⚙️ CONFIGURAÇÕES DE ESTRATÉGIA
HABILITAR_VENDA = False
USAR_NEUTRO = True

# ⚙️ CONFIGURAÇÕES DE ALAVANCAGEM
ALAVANCAGEM_ML = 3
ALAVANCAGEM_BH = 1

# 💰 CONFIGURAÇÕES DE CUSTOS OPERACIONAIS
CUSTOS = {
    'corretagem': 0.0005,
    'slipage': 0.001,
    'emolumentos': 0.00003,
    'taxa_liquidacao': 0.000025,
    'iss': 0.00005,
    'custo_total_por_trade': 0
}

CUSTOS['custo_total_por_trade'] = (
    CUSTOS['corretagem'] * 2 +
    CUSTOS['slipage'] * 2 +
    CUSTOS['emolumentos'] * 2 +
    CUSTOS['taxa_liquidacao'] * 2 +
    CUSTOS['iss']
)

# ⚙️ CONFIGURAÇÕES DE OTIMIZAÇÃO
TRY = 10
INVESTIMENTO_INICIAL = 10000.00
FUT = 3
inferior = 0.35
superior = 0.65
N_SPLITS_OUTER = 5
N_SPLITS_INNER = 3
N_SPLITS_FINAL = 3

# 🎯 TOP MELHORES AÇÕES
TOP_N = 10

TICKERS_DISPONIVEIS = {

    # =========================
    # ETFs Brasil
    # =========================
    'BOVA11.SA':'IBOV',
    'SMAL11.SA':'SMALL',
    'IVVB11.SA':'SP500',
    'DIVO11.SA':'DIVIDENDOS',
    'PIBB11.SA':'IBRX50',
    'XBOV11.SA':'IBOV2',
    'GOVE11.SA':'GOVERNANCA',
    'MATB11.SA':'MATERIAIS',
    'FIND11.SA':'FINANCEIRO',

    # =========================
    # FIIs Papel
    # =========================
    'CPTS11.SA':'CPTS',
    # 'VGIR11.SA':'VGIR',
    # 'KNCR11.SA':'KNCR',
    # 'IRDM11.SA':'IRDM',
    # 'RBRR11.SA':'RBRR',
    # 'MXRF11.SA':'MXRF',

    # # =========================
    # # FIIs Tijolo
    # # =========================
    # 'HGLG11.SA':'HGLG',
    # 'XPLG11.SA':'XPLG',
    # 'BTLG11.SA':'BTLG',
    # 'BRCO11.SA':'BRCO',
    # 'LVBI11.SA':'LVBI',
    # 'VISC11.SA':'VISC',

    # # =========================
    # # Bancos
    # # =========================
    # 'ITUB4.SA':'ITAU',
    # 'BBAS3.SA':'BB',
    # 'BPAC11.SA':'BTG',
    # 'ITSA4.SA':'ITAUSA',
    # 'SANB11.SA':'SANTANDER',
    # 'B3SA3.SA':'B3',

    # # =========================
    # # Energia Elétrica
    # # =========================
    # 'TAEE11.SA':'TAESA',
    # 'EGIE3.SA':'ENGIE',
    # 'CMIG4.SA':'CEMIG',
    # 'ALUP11.SA':'ALUPAR',
    # 'CPFE3.SA':'CPFL',

    # # =========================
    # # Petróleo
    # # =========================
    # 'PETR4.SA':'PETROBRAS',
    # 'PRIO3.SA':'PRIO',
    # 'RRRP3.SA':'3R',

    # # =========================
    # # Mineração / Siderurgia
    # # =========================
    # 'VALE3.SA':'VALE',
    # 'GGBR4.SA':'GERDAU',
    # 'CSNA3.SA':'CSN',
    # 'USIM5.SA':'USIMINAS',

    # # =========================
    # # Consumo
    # # =========================
    # 'ABEV3.SA':'AMBEV',
    # 'RADL3.SA':'RAIADROGASIL',

    # # =========================
    # # Utilities
    # # =========================
    # 'SBSP3.SA':'SABESP',
    # 'SAPR11.SA':'SANEPAR',

    # # =========================
    # # Seguros
    # # =========================
    # 'CXSE3.SA':'CAIXA SEGURIDADE',
    # 'BBSE3.SA':'BB SEGURIDADE',

    # # =========================
    # # Saúde
    # # =========================
    # 'FLRY3.SA':'FLEURY',
    # 'RDOR3.SA':'REDE DOR',

    # # =========================
    # # Tecnologia
    # # =========================
    # 'TOTS3.SA':'TOTVS',

    # # =========================
    # # Papel e Celulose
    # # =========================
    # 'SUZB3.SA':'SUZANO',
    # 'KLBN11.SA':'KLABIN',

    # # =========================
    # # Indústria
    # # =========================
    # 'WEGE3.SA':'WEG',
    # 'RAPT4.SA':'RANDON',
}

# ============================================================================
# FUNÇÕES AUXILIARES
# ============================================================================

def get_walk_forward_splits(df, n_splits):
    """Retorna lista de (treino, validação) para walk-forward"""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    splits = []
    for idx_treino, idx_val in tscv.split(df):
        splits.append((idx_treino, idx_val))
    return splits

def gerar_sinais_trading(predicoes, habilitar_venda=True, usar_neutro=True):
    """Converte previsões do modelo em sinais de trading"""
    sinais = np.zeros(len(predicoes), dtype=int)
    sinais[predicoes == 2] = 1
    if usar_neutro:
        sinais[predicoes == 1] = 0
    else:
        sinais[predicoes == 1] = 0
    if habilitar_venda:
        sinais[predicoes == 0] = -1
    else:
        sinais[predicoes == 0] = 0
    return sinais

def aplicar_custos_operacionais(df_trades, custos):
    """Aplica custos operacionais realistas nos retornos"""
    df = df_trades.copy()

    df['Mudanca_Sinal'] = df['Sinal'].diff().abs() > 0
    df['Trade_Entrada'] = df['Mudanca_Sinal'] & (df['Sinal'] != 0)
    df['Trade_Saida'] = df['Mudanca_Sinal'] & (df['Sinal'].shift(1) != 0)

    df['Custo_Operacao'] = 0.0
    df['Custo_Acumulado'] = 0.0
    custo_acumulado = 0.0

    for i in range(len(df)):
        if df.iloc[i]['Trade_Entrada'] or df.iloc[i]['Trade_Saida']:
            preco_atual = df.iloc[i]['Close']
            custo_trade = preco_atual * custos['custo_total_por_trade']

            if df.iloc[i]['Trade_Entrada'] and not df.iloc[i]['Trade_Saida']:
                custo_trade = custo_trade / 2
            if df.iloc[i]['Trade_Saida'] and not df.iloc[i]['Trade_Entrada']:
                custo_trade = custo_trade / 2

            df.iloc[i, df.columns.get_loc('Custo_Operacao')] = custo_trade
            custo_acumulado += custo_trade

        df.iloc[i, df.columns.get_loc('Custo_Acumulado')] = custo_acumulado

    df['Retorno_ML_Bruto'] = df['Retorno_ML'].copy()
    df['Retorno_ML'] = df['Retorno_ML'] - (df['Custo_Operacao'] / (INVESTIMENTO_INICIAL * ALAVANCAGEM_ML))

    if len(df) > 0:
        custo_inicial_bh = df.iloc[0]['Close'] * custos['custo_total_por_trade'] / 2
        df['Retorno_BH'] = df['Retorno_BH'] - (custo_inicial_bh / INVESTIMENTO_INICIAL)

    return df

def descrever_estrategia(habilitar_venda, usar_neutro, alavancagem_ml, alavancagem_bh, custos):
    """Retorna descrição textual da estratégia configurada"""
    if habilitar_venda and usar_neutro:
        estrategia = "Compra/Venda/Neutro"
    elif habilitar_venda and not usar_neutro:
        estrategia = "Compra/Venda"
    elif not habilitar_venda and usar_neutro:
        estrategia = "Compra/Neutro"
    else:
        estrategia = "Apenas Compra"

    custo_total_pct = custos['custo_total_por_trade'] * 100
    return f"{estrategia} (ML: {alavancagem_ml}x | B&H: {alavancagem_bh}x | Custos: {custo_total_pct:.2f}%/trade)"

def calcular_features_sem_target(dados, roll_curto, roll_longo):
    """Calcula todas as features exceto o target"""
    df = dados[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

    df['ret_1'] = df['Close'].pct_change(1)
    df['ret_5'] = df['Close'].pct_change(5)
    df['ret_10'] = df['Close'].pct_change(roll_curto)
    df['ret_21'] = df['Close'].pct_change(roll_longo)

    df['mm10'] = df['Close'].rolling(roll_curto).mean()
    df['mm21'] = df['Close'].rolling(roll_longo).mean()

    df['vol_10'] = df['ret_1'].rolling(roll_curto).std()
    df['vol_21'] = df['ret_1'].rolling(roll_longo).std()
    df['vol_norm'] = df['vol_10'] / (df['vol_21'] + 1e-6)

    df['driver_sharpe'] = (
        df['ret_10'].rolling(roll_curto).mean() /
        (df['ret_10'].rolling(roll_curto).std() + 1e-4)
    )

    ewm = df['Close'].ewm(span=roll_curto, adjust=False).mean()
    ruido = df['Close'] - ewm

    df['driver_ewm'] = (
        ewm.pct_change() /
        (ruido.rolling(roll_curto).std() + 1e-4)
    )

    df['roc_suave'] = df['ret_10'].rolling(5).mean()

    df['driver_roc'] = (
        df['roc_suave'] /
        (df['ret_10'].rolling(roll_curto).std() + 1e-4)
    )

    df['driver'] = (
        df['driver_sharpe'] * 0.40 +
        df['driver_ewm'] * 0.30 +
        df['driver_roc'] * 0.30
    )

    df['driver'] = np.clip(df['driver'], -3, 3)

    df['cont'] = 0
    cont_plus = cont_down = 0

    for i in df.index:
        if df.loc[i, 'mm10'] < df.loc[i, 'Close']:
            cont_plus += 1
            cont_down = 0
        elif df.loc[i, 'mm10'] > df.loc[i, 'Close']:
            cont_down -= 1
            cont_plus = 0
        else:
            cont_plus = cont_down = 0
        df.loc[i, 'cont'] = cont_plus + cont_down

    df['driver_mom'] = df['driver'].diff(5)
    df['driver_suave'] = df['driver'].rolling(5).mean()
    df['driver_conf'] = df['driver'].abs()

    df['vol_zscore'] = (
        (df['Volume'] - df['Volume'].rolling(roll_curto).mean()) /
        (df['Volume'].rolling(roll_curto).std() + 1e-6)
    )

    df['preco_vs_mm10'] = df['Close'] / df['mm10'] - 1
    df['dist_max_10'] = df['Close'] / df['High'].rolling(roll_curto).max() - 1

    high_roll = df['High'].rolling(roll_curto).max()
    low_roll = df['Low'].rolling(roll_curto).min()

    mov_liquido = (df['Close'] - df['Close'].shift(roll_curto)).abs()
    mov_total = df['Close'].diff().abs().rolling(roll_curto).sum()
    df['trend_eff'] = mov_liquido / (mov_total + 1e-6)

    df['range_pos'] = (
        (df['Close'] - low_roll) /
        (high_roll - low_roll + 1e-6)
    )

    df['dist_topo'] = (
        (high_roll - df['Close']) /
        (df['vol_10'] * df['Close'] + 1e-6)
    )

    df['dist_fundo'] = (
        (df['Close'] - low_roll) /
        (df['vol_10'] * df['Close'] + 1e-6)
    )

    df['mm10_slope'] = (
        (df['mm10'] - df['mm10'].shift(5)) /
        (df['vol_10'] * df['Close'] + 1e-6)
    )

    df['mm_ratio'] = df['mm10'] / (df['mm21'] + 1e-6)

    df['body_ratio'] = (
        (df['Close'] - df['Open']) /
        (df['High'] - df['Low'] + 1e-6)
    )

    df['buy_pressure'] = (
        (df['Close'] - df['Low']) /
        (df['High'] - df['Low'] + 1e-6)
    )

    df['sell_pressure'] = (
        (df['High'] - df['Close']) /
        (df['High'] - df['Low'] + 1e-6)
    )

    df['gap'] = (
        (df['Open'] - df['Close'].shift(1)) /
        (df['vol_10'] * df['Close'] + 1e-6)
    )

    df['ret_acc'] = df['ret_5'] - df['ret_21']
    df['vol_change'] = df['vol_norm'].diff(5)

    df['trend_persist'] = (
        np.sign(df['ret_1'])
        .rolling(roll_curto)
        .mean()
    )

    df['ret_skew'] = df['ret_1'].rolling(roll_curto).skew()
    df['ret_kurt'] = df['ret_1'].rolling(roll_curto).kurt()

    ret = df['Close'] / df['Close'].shift(roll_curto) - 1
    draw = df['Close'] / df['Close'].rolling(roll_curto).max() - 1
    df['mom_draw'] = ret / (draw.abs() + 0.01)

    up = df['ret_1'].clip(lower=0).rolling(roll_curto).std()
    down = df['ret_1'].clip(upper=0).rolling(roll_curto).std()
    df['vol_direction'] = up / (down + 1e-6)

    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df

FEATURES = [
    'ret_1','ret_5','ret_10','ret_21',
    'driver','driver_mom','driver_suave','driver_conf',
    'cont','vol_10','vol_norm','vol_zscore',
    'preco_vs_mm10','dist_max_10',
    'trend_eff','range_pos','dist_topo','dist_fundo',
    'mm10_slope','mm_ratio','body_ratio','buy_pressure',
    'sell_pressure','gap','ret_acc','vol_change',
    'trend_persist','ret_skew','ret_kurt','mom_draw','vol_direction'
]

# ============================================================================
# ESPAÇO DE HIPERPARÂMETROS
# ============================================================================

espaco_parametros = [
    Categorical(['10y'], name='time_period'),
    Integer(5, 20, name='roll_curto'),
    Integer(16, 40, name='roll_longo'),
    Integer(50, 300, name='n_estimators'),
    Integer(2, 5, name='max_depth'),
    Real(0.01, 0.10, name='learning_rate'),
    Integer(5, 25, name='min_child_weight'),
    Real(1.0, 50.0, name='reg_alpha'),
    Real(5.0, 100.0, name='reg_lambda'),
    Real(0.1, 2.0, name='gamma'),
    Real(0.40, 0.80, name='subsample'),
    Real(0.30, 0.70, name='colsample_bytree'),
]

# ============================================================================
# CLASSE PARA GERENCIAR MÚLTIPLAS AÇÕES
# ============================================================================

class OtimizadorMultiAtivos:
    def __init__(self):
        self.resultados = {}
        self.top_10 = []
        self.modelos_top = {}
        self.df_consolidado = None

    def criar_funcao_objetivo(self, ticker):
        """Cria uma função objetivo específica para cada ticker"""

        @use_named_args(espaco_parametros)
        def avaliar_configuracao(**params):
            time_period = params['time_period']
            roll_curto = int(params['roll_curto'])
            roll_longo = int(params['roll_longo'])

            if roll_curto >= roll_longo:
                return 5.0
            if roll_longo < roll_curto * 2:
                return 4.0

            try:
                raw = yf.Ticker(ticker).history(period=time_period, interval="1d").reset_index()
                if len(raw) < 252:
                    return 3.0

                df = calcular_features_sem_target(raw, roll_curto, roll_longo)
                if len(df) < 100:
                    return 2.0

                base = XGBClassifier(
                    n_estimators=int(params['n_estimators']),
                    max_depth=int(params['max_depth']),
                    learning_rate=params['learning_rate'],
                    min_child_weight=int(params['min_child_weight']),
                    reg_alpha=params['reg_alpha'],
                    reg_lambda=params['reg_lambda'],
                    gamma=params['gamma'],
                    subsample=params['subsample'],
                    colsample_bytree=params['colsample_bytree'],
                    random_state=42,
                    objective='multi:softprob',
                    num_class=3,
                    eval_metric='mlogloss',
                    verbosity=0
                )

                todos_retornos = []
                splits = get_walk_forward_splits(df, N_SPLITS_OUTER)

                for idx_treino, idx_val in splits:
                    if len(idx_treino) < 50 or len(idx_val) < 20:
                        continue

                    df_treino = df.iloc[idx_treino].copy()
                    df_val = df.iloc[idx_val].copy()

                    driver_fut_treino = df_treino['driver'].shift(-FUT) - df_treino['driver']
                    limite_venda = driver_fut_treino.quantile(inferior)
                    limite_compra = driver_fut_treino.quantile(superior)

                    df_treino['Alvo'] = 1
                    df_treino.loc[driver_fut_treino >= limite_compra, 'Alvo'] = 2
                    df_treino.loc[driver_fut_treino <= limite_venda, 'Alvo'] = 0

                    driver_fut_val = df_val['driver'].shift(-FUT) - df_val['driver']
                    df_val['Alvo'] = 1
                    df_val.loc[driver_fut_val >= limite_compra, 'Alvo'] = 2
                    df_val.loc[driver_fut_val <= limite_venda, 'Alvo'] = 0

                    df_treino = df_treino.dropna(subset=['Alvo'])
                    df_val = df_val.dropna(subset=['Alvo'])

                    if len(df_treino) < 30 or len(df_val) < 10:
                        continue

                    X_treino = df_treino[FEATURES]
                    y_treino = df_treino['Alvo']
                    X_val = df_val[FEATURES]
                    y_val = df_val['Alvo']

                    tscv_calib = TimeSeriesSplit(n_splits=min(N_SPLITS_INNER, 3))
                    modelo = CalibratedClassifierCV(base, method='isotonic', cv=tscv_calib)
                    modelo.fit(X_treino, y_treino)

                    sinal_pred = modelo.predict(X_val)
                    sinal_trading = gerar_sinais_trading(sinal_pred, HABILITAR_VENDA, USAR_NEUTRO)

                    val_data = df_val.copy()
                    val_data['Sinal'] = sinal_trading
                    val_data['Retorno_Real'] = val_data['Close'].pct_change()
                    val_data['Retorno_ML'] = val_data['Sinal'].shift(1) * val_data['Retorno_Real'] * ALAVANCAGEM_ML
                    val_data['Retorno_BH'] = val_data['Retorno_Real'] * ALAVANCAGEM_BH

                    val_data = aplicar_custos_operacionais(val_data, CUSTOS)
                    todos_retornos.append(val_data['Retorno_ML'].dropna())

                if not todos_retornos:
                    return 1.0

                todos_retornos = pd.concat(todos_retornos)
                retorno_medio = todos_retornos.mean()
                std_retorno = todos_retornos.std()

                if std_retorno > 0:
                    sharpe = (retorno_medio / std_retorno) * np.sqrt(252)
                else:
                    sharpe = -5

                num_sinais = len(todos_retornos[todos_retornos != 0])
                total_periodos = len(todos_retornos)
                taxa_sinal = num_sinais / total_periodos if total_periodos > 0 else 0

                if taxa_sinal < 0.10:
                    sharpe -= 2.0
                elif taxa_sinal > 0.90:
                    sharpe -= 1.0

                return -sharpe

            except Exception as e:
                return 10.0

        return avaliar_configuracao

    def otimizar_acao(self, ticker, nome_ativo):
        """Otimiza uma única ação"""
        print(f"\n{'='*72}")
        print(f"  OTIMIZANDO: {nome_ativo} ({ticker})")
        print(f"{'='*72}")

        funcao_objetivo = self.criar_funcao_objetivo(ticker)

        res_busca = gp_minimize(
            funcao_objetivo,
            espaco_parametros,
            n_calls=TRY,
            random_state=42,
            verbose=False
        )

        melhores_params = dict(zip([p.name for p in espaco_parametros], res_busca.x))
        sharpe_otimo = -res_busca.fun

        print(f"  ✅ Sharpe Ótimo: {sharpe_otimo:.3f}")

        # Treinar modelo final
        modelo_final = self.treinar_modelo_final(ticker, nome_ativo, melhores_params)

        self.resultados[ticker] = {
            'nome': nome_ativo,
            'sharpe': sharpe_otimo,
            'params': melhores_params,
            'modelo': modelo_final
        }

        return sharpe_otimo, modelo_final

    def treinar_modelo_final(self, ticker, nome_ativo, params):
        """Treina o modelo final com os melhores parâmetros"""
        time_period = params['time_period']
        roll_curto = int(params['roll_curto'])
        roll_longo = int(params['roll_longo'])

        raw = yf.Ticker(ticker).history(period=time_period, interval="1d").reset_index()
        df = calcular_features_sem_target(raw, roll_curto, roll_longo)

        tscv = TimeSeriesSplit(n_splits=N_SPLITS_FINAL)
        splits = list(tscv.split(df))
        idx_treino, idx_teste = splits[-1]

        df_treino = df.iloc[idx_treino].copy()
        df_teste = df.iloc[idx_teste].copy()

        driver_fut_treino = df_treino['driver'].shift(-FUT) - df_treino['driver']
        limite_venda = driver_fut_treino.quantile(inferior)
        limite_compra = driver_fut_treino.quantile(superior)

        df_treino['Alvo'] = 1
        df_treino.loc[driver_fut_treino >= limite_compra, 'Alvo'] = 2
        df_treino.loc[driver_fut_treino <= limite_venda, 'Alvo'] = 0

        driver_fut_teste = df_teste['driver'].shift(-FUT) - df_teste['driver']
        df_teste['Alvo'] = 1
        df_teste.loc[driver_fut_teste >= limite_compra, 'Alvo'] = 2
        df_teste.loc[driver_fut_teste <= limite_venda, 'Alvo'] = 0

        df_treino = df_treino.dropna(subset=['Alvo'])
        df_teste = df_teste.dropna(subset=['Alvo'])

        X_treino = df_treino[FEATURES]
        y_treino = df_treino['Alvo']
        X_teste = df_teste[FEATURES]
        y_teste = df_teste['Alvo']

        base = XGBClassifier(
            n_estimators=int(params['n_estimators']),
            max_depth=int(params['max_depth']),
            learning_rate=params['learning_rate'],
            min_child_weight=int(params['min_child_weight']),
            reg_alpha=params['reg_alpha'],
            reg_lambda=params['reg_lambda'],
            gamma=params['gamma'],
            subsample=params['subsample'],
            colsample_bytree=params['colsample_bytree'],
            random_state=42,
            objective='multi:softprob',
            num_class=3,
            eval_metric='mlogloss',
            verbosity=0
        )

        tscv_calib = TimeSeriesSplit(n_splits=N_SPLITS_FINAL)
        modelo = CalibratedClassifierCV(base, method='isotonic', cv=tscv_calib)
        modelo.fit(X_treino, y_treino)

        # Previsões e sinais
        pred_teste = modelo.predict(X_teste)
        sinal_trading = gerar_sinais_trading(pred_teste, HABILITAR_VENDA, USAR_NEUTRO)

        # Backtest
        testes = df_teste.copy().reset_index(drop=True)
        testes['Sinal'] = sinal_trading
        testes['Classe'] = pred_teste
        testes['Retorno_Real'] = testes['Close'].pct_change()
        testes['Retorno_ML'] = testes['Sinal'].shift(1) * testes['Retorno_Real'] * ALAVANCAGEM_ML
        testes['Retorno_BH'] = testes['Retorno_Real'] * ALAVANCAGEM_BH
        testes = aplicar_custos_operacionais(testes, CUSTOS)
        testes_validos = testes.dropna(subset=['Retorno_ML'])

        return {
            'modelo': modelo,
            'params': params,
            'limites': {'venda': float(limite_venda), 'compra': float(limite_compra)},
            'backtest': testes_validos,
            'df_teste': df_teste
        }

    def otimizar_todas_acoes(self):
        """Otimiza todas as ações da lista"""
        print(f"\n{'='*72}")
        print(f"  INICIANDO OTIMIZAÇÃO DE {len(TICKERS_DISPONIVEIS)} ATIVOS")
        print(f"{'='*72}")
        print(f"  Estratégia: {descrever_estrategia(HABILITAR_VENDA, USAR_NEUTRO, ALAVANCAGEM_ML, ALAVANCAGEM_BH, CUSTOS)}")

        resultados_sharpe = []

        for ticker, nome in tqdm(TICKERS_DISPONIVEIS.items(), desc="Otimizando ativos"):
            try:
                sharpe, modelo = self.otimizar_acao(ticker, nome)
                resultados_sharpe.append((ticker, nome, sharpe))
            except Exception as e:
                print(f"  ❌ Erro ao otimizar {nome} ({ticker}): {str(e)[:100]}")
                continue

        # Ordenar por Sharpe e selecionar top 10
        resultados_sharpe.sort(key=lambda x: x[2], reverse=True)
        self.top_10 = resultados_sharpe[:TOP_N]

        print(f"\n{'='*72}")
        print(f"  TOP {TOP_N} MELHORES ATIVOS")
        print(f"{'='*72}")
        for i, (ticker, nome, sharpe) in enumerate(self.top_10, 1):
            print(f"  {i:2d}. {nome:15} ({ticker:12}) - Sharpe: {sharpe:.3f}")

        return self.top_10

    def criar_backtest_consolidado(self):
        """
        Gera o backtest consolidado simulando a evolução patrimonial real (em dinheiro)
        de cada ativo da cesta, removendo vieses de rebalanceamento artificial.
        """
        # Dicionários para guardar o valor financeiro diário de cada ativo
        valor_ml = {}
        valor_bh = {}

        # Lista com os ativos presentes na cesta (ex: top_10)
        ativos_cesta = self.top_10  # Ajuste o nome da variável se na sua classe for diferente
        INVESTIMENTO_POR_ACAO = 200.00  # Define um capital inicial padrão por ativo

        for ticker in ativos_cesta:
            # Pega o DataFrame específico deste ativo criado nas etapas anteriores
            df_ativo = self.df_ativos_individuais[ticker].copy()

            # 1. GARANTIR CÁLCULO CORRETO DOS RETORNOS (Sem prever o futuro)
            df_ativo['Retorno_Real'] = df_ativo['Close'].pct_change()

            # 2. ALINHAR O SINAL DE HOJE COM O RETORNO DE AMANHÃ (.shift(1))
            # O sinal de ontem opera sobre o retorno real de hoje
            df_ativo['Retorno_ML_Diario'] = df_ativo['Sinal'].shift(1) * df_ativo['Retorno_Real'] * ALAVANCAGEM_ML
            df_ativo['Retorno_BH_Diario'] = df_ativo['Retorno_Real'] * ALAVANCAGEM_BH

            # Preenche os NaN iniciais com 0 para não quebrar o produtório
            df_ativo['Retorno_ML_Diario'] = df_ativo['Retorno_ML_Diario'].fillna(0)
            df_ativo['Retorno_BH_Diario'] = df_ativo['Retorno_BH_Diario'].fillna(0)

            # 3. CALCULAR A EVOLUÇÃO PATRIMONIAL DIÁRIA (EM DINHEIRO)
            # Capital Inicial * (1 + Retorno_Diario).cumprod()
            valor_ml[ticker] = INVESTIMENTO_POR_ACAO * (1 + df_ativo['Retorno_ML_Diario']).cumprod()
            valor_bh[ticker] = INVESTIMENTO_POR_ACAO * (1 + df_ativo['Retorno_BH_Diario']).cumprod()

        # Converte os dicionários patrimoniais para DataFrames indexados pela Data
        df_valor_ml = pd.DataFrame(valor_ml)
        df_valor_bh = pd.DataFrame(valor_bh)

        # Soma o valor em dinheiro de todas as ações dia após dia para ter a curva da cesta
        valor_total_ml = df_valor_ml.sum(axis=1)
        valor_total_bh = df_valor_bh.sum(axis=1)

        # 4. CALCULAR AS MÉTRICAS ACUMULADAS FINAIS REAIS
        investimento_inicial_total = INVESTIMENTO_POR_ACAO * len(ativos_cesta)

        ret_acum_ml = (valor_total_ml.iloc[-1] / investimento_inicial_total) - 1
        ret_acum_bh = (valor_total_bh.iloc[-1] / investimento_inicial_total) - 1

        # 5. CÁLCULO DOS RETORNOS DIÁRIOS DA CARTEIRA TOTAL PARA O SHARPE RATIO
        # (Variação percentual do patrimônio total de um dia para o outro)
        retornos_carteira_ml = valor_total_ml.pct_change().dropna()
        retornos_carteira_bh = valor_total_bh.pct_change().dropna()

        # Sharpe Ratio (Anualizado, assumindo 252 dias úteis)
        sharpe_ml = 0 if retornos_carteira_ml.std() == 0 else (retornos_carteira_ml.mean() / retornos_carteira_ml.std()) * np.sqrt(252)
        sharpe_bh = 0 if retornos_carteira_bh.std() == 0 else (retornos_carteira_bh.mean() / retornos_carteira_bh.std()) * np.sqrt(252)

        # Retorna o dicionário com as métricas reais idênticas à lógica do RODAR_MODELO_2
        metricas = {
            'ret_acum_ml': ret_acum_ml,
            'ret_acum_bh': ret_acum_bh,
            'sharpe_ml': sharpe_ml,
            'sharpe_bh': sharpe_bh,
            'valor_total_ml': valor_total_ml, # Útil se quiser usar para plotar o gráfico depois
            'valor_total_bh': valor_total_bh
        }

        return metricas

    def plotar_resultados(self):
        """Plota gráficos dos resultados"""
        fig = plt.figure(figsize=(24, 18))
        fig.patch.set_facecolor('#0E1117')
        gs = gridspec.GridSpec(3, 3, figure=fig, wspace=0.3, hspace=0.4)

        PANEL_BG = '#161B22'
        TEXTO = '#E5E7EB'

        estrategia_desc = descrever_estrategia(HABILITAR_VENDA, USAR_NEUTRO, ALAVANCAGEM_ML, ALAVANCAGEM_BH, CUSTOS)

        # 1. Ranking de Sharpe
        ax1 = fig.add_subplot(gs[0, :])
        ax1.set_facecolor(PANEL_BG)

        nomes = [f"{n}\n({t})" for t, n, s in self.top_10]
        sharpes = [s for t, n, s in self.top_10]
        cores = plt.cm.RdYlGn(np.array(sharpes) / max(sharpes))

        bars = ax1.bar(range(len(nomes)), sharpes, color=cores, edgecolor='white', linewidth=0.5)
        ax1.set_xticks(range(len(nomes)))
        ax1.set_xticklabels(nomes, color=TEXTO, fontsize=8)
        ax1.set_title(f'Top {TOP_N} Ativos por Sharpe Ratio\n{estrategia_desc}',
                     color=TEXTO, fontsize=12, fontweight='bold')
        ax1.set_ylabel('Sharpe Ratio', color=TEXTO)
        ax1.tick_params(colors=TEXTO)
        ax1.grid(True, alpha=0.1, axis='y')

        for bar, val in zip(bars, sharpes):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.2f}', ha='center', va='bottom', color=TEXTO, fontsize=8)

        # 2. Evolução Patrimonial Consolidada
        ax2 = fig.add_subplot(gs[1, :2])
        ax2.set_facecolor(PANEL_BG)

        met = self.metricas_consolidadas
        ax2.plot(met['valor_ml'].index, met['valor_ml'].values, color='#2F9FD2', lw=2.5,
                label=f'Cesta ML (Sharpe: {met["sharpe_ml"]:.2f})')
        ax2.plot(met['valor_bh'].index, met['valor_bh'].values, color='#6B7280', lw=1.5, ls='--',
                label=f'Cesta B&H (Sharpe: {met["sharpe_bh"]:.2f})')
        ax2.axhline(y=INVESTIMENTO_INICIAL, color='white', lw=0.5, ls=':', alpha=0.5)

        ax2.set_title(f'Cesta Consolidada - Evolução Patrimonial\nInvestimento Inicial: R${INVESTIMENTO_INICIAL:,.2f}',
                     color=TEXTO, fontsize=12, fontweight='bold')
        ax2.tick_params(colors=TEXTO, labelsize=9)
        ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
        ax2.grid(True, alpha=0.1)
        ax2.legend(facecolor=PANEL_BG, labelcolor=TEXTO, fontsize=9)

        # 3. Drawdown Consolidado
        ax3 = fig.add_subplot(gs[1, 2])
        ax3.set_facecolor(PANEL_BG)

        dd_ml_serie = (met['valor_ml'] / met['valor_ml'].cummax() - 1) * 100
        dd_bh_serie = (met['valor_bh'] / met['valor_bh'].cummax() - 1) * 100

        ax3.fill_between(range(len(dd_ml_serie)), dd_ml_serie.values, 0,
                        color='#EF4444', alpha=0.4, label=f'ML (Max: {met["dd_ml"]:.1%})')
        ax3.fill_between(range(len(dd_bh_serie)), dd_bh_serie.values, 0,
                        color='#6B7280', alpha=0.2, label=f'B&H (Max: {met["dd_bh"]:.1%})')

        ax3.set_title('Drawdown Consolidado (%)', color=TEXTO, fontsize=11, fontweight='bold')
        ax3.tick_params(colors=TEXTO, labelsize=8)
        ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}%'))
        ax3.grid(True, alpha=0.1)
        ax3.legend(facecolor=PANEL_BG, labelcolor=TEXTO, fontsize=8)

        # 4. Retornos Individuais
        ax4 = fig.add_subplot(gs[2, :])
        ax4.set_facecolor(PANEL_BG)

        # Calcular retornos individuais
        retornos_ind = []
        for ticker, nome, sharpe in self.top_10:
            modelo_data = self.resultados[ticker]['modelo']
            df_bt = modelo_data['backtest']
            ret_acum = (1 + df_bt['Retorno_ML'].dropna()).prod() - 1
            retornos_ind.append((nome, ret_acum * 100))

        nomes = [n for n, r in retornos_ind]
        rets = [r for n, r in retornos_ind]
        cores = ['#22C55E' if r > 0 else '#EF4444' for r in rets]

        bars = ax4.barh(range(len(nomes)), rets, color=cores, alpha=0.8, edgecolor='white', linewidth=0.5)
        ax4.set_yticks(range(len(nomes)))
        ax4.set_yticklabels(nomes, color=TEXTO, fontsize=9)
        ax4.set_title('Retorno Individual de Cada Ativo (%)', color=TEXTO, fontsize=12, fontweight='bold')
        ax4.tick_params(colors=TEXTO)
        ax4.axvline(x=0, color='white', lw=0.5, ls='-', alpha=0.5)
        ax4.grid(True, alpha=0.1, axis='x')
        ax4.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}%'))

        for bar, val in zip(bars, rets):
            ax4.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%', va='center', color=TEXTO, fontsize=8)

        plt.suptitle(f'Análise da Cesta de Ativos - Top {TOP_N}\n{estrategia_desc}',
                    color=TEXTO, fontsize=14, fontweight='bold', y=0.98)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        estrategia_tag = f"CESTA_TOP{TOP_N}"

        os.makedirs('outputs', exist_ok=True)
        plt.savefig(f'outputs/cesta_consolidada_{estrategia_tag}_{timestamp}.png',
                   dpi=150, bbox_inches='tight', facecolor='#0E1117')
        plt.show()

        return fig

    def exportar_modelo_consolidado(self):
        """Exporta modelo consolidado para operar a cesta de ativos"""
        print(f"\n{'='*72}")
        print(f"  EXPORTANDO MODELO CONSOLIDADO")
        print(f"{'='*72}")

        modelo_consolidado = {
            'tipo': 'cesta_ml',
            'estrategia': {
                'habilitar_venda': HABILITAR_VENDA,
                'usar_neutro': USAR_NEUTRO,
                'alavancagem_ml': ALAVANCAGEM_ML,
                'alavancagem_bh': ALAVANCAGEM_BH,
                'custos': CUSTOS
            },
            'top_n': TOP_N,
            'metricas_consolidadas': converter_para_python(self.metricas_consolidadas),
            'ativos': {}
        }

        for ticker, nome, sharpe in self.top_10:
            modelo_data = self.resultados[ticker]['modelo']
            modelo_consolidado['ativos'][ticker] = {
                'nome': nome,
                'sharpe': sharpe,
                'modelo': modelo_data['modelo'],
                'params': modelo_data['params'],
                'limites': modelo_data['limites'],
                'peso': 1.0 / TOP_N
            }

        # Salvar
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        estrategia_tag = f"CESTA_TOP{TOP_N}"

        os.makedirs('modelos', exist_ok=True)

        # Modelo principal
        caminho_modelo = f'modelos/cesta_consolidada_{estrategia_tag}_{timestamp}.pkl'
        joblib.dump(modelo_consolidado, caminho_modelo)
        print(f"✅ Modelo consolidado: {caminho_modelo}")

        # Configuração JSON
        config_json = {
            'timestamp': timestamp,
            'estrategia': modelo_consolidado['estrategia'],
            'top_n': TOP_N,
            'ativos': {
                ticker: {
                    'nome': info['nome'],
                    'sharpe': float(info['sharpe']),
                    'limites': info['limites'],
                    'peso': float(info['peso'])
                }
                for ticker, info in modelo_consolidado['ativos'].items()
            }
        }

        caminho_json = f'modelos/config_cesta_{estrategia_tag}_{timestamp}.json'
        with open(caminho_json, 'w', encoding='utf-8') as f:
            json.dump(config_json, f, indent=2, ensure_ascii=False)
        print(f"✅ Configuração JSON: {caminho_json}")

        return modelo_consolidado

# ============================================================================
# EXECUÇÃO PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    # Criar otimizador
    otimizador = OtimizadorMultiAtivos()

    # Otimizar todas as ações
    top_10 = otimizador.otimizar_todas_acoes()

    # Criar backtest consolidado
    metricas = otimizador.criar_backtest_consolidado()

    # Plotar resultados
    fig = otimizador.plotar_resultados()

    # Exportar modelo consolidado
    modelo_consolidado = otimizador.exportar_modelo_consolidado()

    # Resumo final
    print(f"\n{'='*72}")
    print(f"  RESUMO FINAL - CESTA CONSOLIDADA")
    print(f"{'='*72}")
    print(f"  📊 Ativos na cesta: {TOP_N}")
    print(f"  ⚙️  Estratégia: {descrever_estrategia(HABILITAR_VENDA, USAR_NEUTRO, ALAVANCAGEM_ML, ALAVANCAGEM_BH, CUSTOS)}")
    print(f"  🎯 Sharpe ML Consolidado: {metricas['sharpe_ml']:.2f}")
    print(f"  🎯 Sharpe B&H Consolidado: {metricas['sharpe_bh']:.2f}")
    print(f"  💰 Retorno ML: {metricas['ret_acum_ml']:.1%}")
    print(f"  💰 Retorno B&H: {metricas['ret_acum_bh']:.1%}")
    print(f"  📉 Drawdown ML: {metricas['dd_ml']:.1%}")
    print(f"  📉 Drawdown B&H: {metricas['dd_bh']:.1%}")
    print(f"  💸 Excesso de Retorno: {metricas['ret_acum_ml'] - metricas['ret_acum_bh']:.1%}")
    print(f"\n  🏆 TOP {TOP_N} ATIVOS:")
    for i, (ticker, nome, sharpe) in enumerate(top_10, 1):
        print(f"     {i:2d}. {nome:15} ({ticker}) - Sharpe: {sharpe:.3f}")

    print(f"\n✅ Processo completo finalizado!")
    print(f"   Modelo consolidado salvo em: modelos/cesta_consolidada_CESTA_TOP{TOP_N}_*.pkl")